# Task05：固定配额随机化与三档邻居暴露

**D2 × E2 | 可从头运行的独立 Identification 演示**

公司在固定的合作网络上均匀抽取恰好 $m$ 人获得 AI。我们关心某名员工自身是否获得 AI，
以及邻居获得 AI 的比例处于低、中、高哪一档时，按该实验设计标准化的潜在结果均值。

本演示顺序：业务问题 → 设定 → 支持概率 → POINT 推导 → IdentificationResult → 穷举验证。
选择 Python 3 内核并 Run All，无需数据文件、模型权重或 GPU。

In [1]:
from pathlib import Path
import sys
from fractions import Fraction
from itertools import combinations
from pprint import pprint

# Support launching from the repository root, task05, or its workspace parent.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "CFM-identification"]
root = next((p for p in candidates if (p / "student_work/task05/support.py").is_file()), None)
if root is None:
    raise RuntimeError("Please open this notebook inside the CFM-identification repository.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
sys.dont_write_bytecode = True
from student_work.task05.support import compute_support, exposure_state
from student_work.task05.rule import identify, REQUIRED_ASSUMPTIONS
print("Task05 modules imported successfully.")

Task05 modules imported successfully.


## 1. 设定：六名员工，恰好两人获得 AI

节点 0 的合作邻居为 1、2、3，因此 $d_0=3$。其他边为 1—4、2—5。
所有 $\binom62=15$ 个合法完整分配等概率，各自权重 $1/15$。

E2：$K_i/d_i<1/3$ 为低；$1/3\le K_i/d_i<2/3$ 为中；$K_i/d_i\ge2/3$ 为高。
故节点 0 的 $K=0$ 为低，$K=1$ 为中，$K=2,3$ 为高。
不同于独立 Bernoulli，本人拿到 AI 后其余人只剩一个名额。

In [2]:
n, m = 6, 2
edges = [(0, 1), (0, 2), (0, 3), (1, 4), (2, 5)]
graph = [[0] * n for _ in range(n)]
for a, b in edges:
    graph[a][b] = graph[b][a] = 1
print("Edges:", edges)
print("Degrees:", [sum(row) for row in graph])
print("Exposure at node 0 for K=0..3:", [exposure_state(k, 3) for k in range(4)])

Edges: [(0, 1), (0, 2), (0, 3), (1, 4), (2, 5)]
Degrees: [3, 2, 2, 1, 1, 1]
Exposure at node 0 for K=0..3: [0, 1, 2, 2]


## 2. 支持概率：必须按完整分配律计数

$$\pi_i(t,s)=\frac{1}{\binom Nm}\sum_{k\in B_s(d_i)}
\binom{d_i}{k}\binom{N-1-d_i}{m-t-k}.$$

分子分别从邻居中选 $k$ 人、从其他非本人节点中选 $m-t-k$ 人。非法组合贡献为零。
计数与概率均精确计算，不使用模拟频率替代 population support。

In [3]:
tables = compute_support(graph, m)
print("node | t | low | medium | high")
for i, table in tables.items():
    for t in (0, 1):
        print(i, t, *(str(table[t, s]) for s in range(3)), sep=" | ")
    assert sum(table.values()) == 1
    assert sum(table[1, s] for s in range(3)) == Fraction(m, n)
assert tables[0][1, 2] == 0
print("All rows normalized; own-treatment marginals equal m/N.")

node | t | low | medium | high
0 | 0 | 1/15 | 2/5 | 1/5
0 | 1 | 2/15 | 1/5 | 0
1 | 0 | 1/5 | 2/5 | 1/15
1 | 1 | 1/5 | 2/15 | 0
2 | 0 | 1/5 | 2/5 | 1/15
2 | 1 | 1/5 | 2/15 | 0
3 | 0 | 2/5 | 0 | 4/15
3 | 1 | 4/15 | 0 | 1/15
4 | 0 | 2/5 | 0 | 4/15
4 | 1 | 4/15 | 0 | 1/15
5 | 0 | 2/5 | 0 | 4/15
5 | 1 | 4/15 | 0 | 1/15
All rows normalized; own-treatment marginals equal m/N.


节点 0 的六个概率为：

| 本人处理 | 低 | 中 | 高 |
|---|---:|---:|---:|
| 0 | 1/15 | 6/15 | 3/15 |
| 1 | 2/15 | 3/15 | 0 |

最后一格不可能出现：本人已占一个名额，三个邻居最多一人被处理，达不到高档的至少两人。
因此不能将联合概率分解为 $P(T_i=t)P(S_i=s)$。

In [4]:
p_high = tables[0][0, 2] + tables[0][1, 2]
incorrect_independence_product = Fraction(m, n) * p_high
print("Actual P(T0=1,S0=high):", tables[0][1, 2])
print("Incorrect product of marginals:", incorrect_independence_product)
assert tables[0][1, 2] != incorrect_independence_product

Actual P(T0=1,S0=high): 0
Incorrect product of marginals: 1/15


## 3. 目标和 POINT 推导

记 $\mathcal U_{i,t,s}=\{u:u_i=t,S_i(u)=s\}$，$q(u)=\mathbf1\{\sum u=m\}/\binom Nm$。
对正支持 arm，目标是

$$\mu_i(t,s)=\sum_{u\in\mathcal U_{i,t,s}}\frac{q(u)}{\pi_i(t,s)}
\mathbb E[Y_i(u)\mid A,X].$$

假设：已知完整分配律；固定处理前网络和背景；完整分配向量随机化；consistency；相关条件绝对一阶矩有限。
加上所查询 arm 的正支持，由全期望、consistency 和 randomization 得：

$$\begin{aligned}
\mathbb E[Y_i^{obs}\mid A,X,T_i=t,S_i=s]
&=\sum_{u\in\mathcal U_{i,t,s}:q(u)>0}\frac{q(u)}{\pi_i(t,s)}
\mathbb E[Y_i^{obs}\mid A,X,T=u]\\
&=\sum_{u\in\mathcal U_{i,t,s}:q(u)>0}\frac{q(u)}{\pi_i(t,s)}
\mathbb E[Y_i(u)\mid A,X]=\mu_i(t,s).
\end{aligned}$$

所以目标由总体观测分布唯一确定，即 POINT。这不要求 E2 是 outcome 的充分暴露映射，
也不要求各人的处理相互独立。零支持时目标的归一化无定义，拒绝 query。
有限样本某格无人不等于总体零支持。

## 4. IdentificationResult：成功与停止原因

以下假设仅为本演示的合成随机实验作显式声明，不是由数据自动验证。
`rule.IdentificationResult` 是本任务本地结果类，不是主仓库同名公共类。
本例给出可识别的总体条件均值表达式，不计算数值估计；未接入 core verifier 和 Estimation handoff，
所以 `verification_status="NOT_RUN"`，`estimation_request=None`。

In [5]:
assumptions = dict.fromkeys(REQUIRED_ASSUMPTIONS, "ADMITTED")
identified = identify(graph, m, node=0, t=0, s=1, assumptions=assumptions)
pprint(identified.to_dict(), sort_dicts=False)
assert identified.status == "POINT"
assert identified.support == Fraction(2, 5)
assert identified.functional is not None
assert identified.estimation_request is None

{'node': 0,
 'treatment': 0,
 'exposure': 1,
 'n': 6,
 'quota': 2,
 'adjacency': ((0, 1, 1, 1, 0, 0),
               (1, 0, 0, 0, 1, 0),
               (1, 0, 0, 0, 0, 1),
               (1, 0, 0, 0, 0, 0),
               (0, 1, 0, 0, 0, 0),
               (0, 0, 1, 0, 0, 0)),
 'assumptions': (('known_assignment_law', 'ADMITTED'),
                 ('full_vector_randomization', 'ADMITTED'),
                 ('consistency', 'ADMITTED'),
                 ('fixed_pretreatment_network', 'ADMITTED'),
                 ('finite_conditional_first_moments', 'ADMITTED')),
 'support': '2/5',
 'status': 'POINT',
 'workflow_status': 'COMPLETE',
 'reason_code': 'DESIGN_STANDARDIZED_ARM_IDENTIFIED',
 'functional': 'E[Y_0^obs | A, X, T_0=0, S_0=1]',
 'verification_status': 'NOT_RUN',
 'estimation_request': None,
 'scope': 'standalone_task05_not_registered_with_core_verifier'}


In [6]:
zero = identify(graph, m, 0, 1, 2, assumptions=assumptions)
missing = identify(graph, m, 0, 0, 1)
rejected = identify(graph, m, 0, 0, 1,
                    assumptions={**assumptions, "consistency": "CONTRADICTED"})
isolated = identify([[0]], 0, 0, 0, 0, assumptions=assumptions)
for label, result in [("zero support", zero), ("missing assumptions", missing),
                      ("rejected assumption", rejected), ("isolated node", isolated)]:
    print(label, result.status, result.workflow_status, result.reason_code, sep=" | ")
    assert result.functional is None
assert missing.workflow_status == "NEEDS_INPUT"
assert zero.reason_code == "ZERO_POPULATION_SUPPORT"
assert isolated.support is None

zero support | UNSUPPORTED | UNSUPPORTED | ZERO_POPULATION_SUPPORT
missing assumptions | UNSUPPORTED | NEEDS_INPUT | REQUIRED_ASSUMPTIONS_NOT_ADMITTED
rejected assumption | UNSUPPORTED | UNSUPPORTED | REQUIRED_ASSUMPTION_REJECTED
isolated node | UNSUPPORTED | UNSUPPORTED | EXPOSURE_UNDEFINED_ISOLATE


## 5. 独立穷举对拍

枚举全部 15 个合法完整分配，每个赋予 1/15 的权重。
这里使用 `Fraction(K,d)` 与分数阈值直接比较，独立于 support.py 的组合数计算和整数分档实现。

In [7]:
allocations = list(combinations(range(n), m))
oracle = {i: {(t, s): Fraction(0) for t in (0, 1) for s in range(3)} for i in range(n)}
for selected_tuple in allocations:
    selected = set(selected_tuple)
    for i in range(n):
        share = Fraction(sum(graph[i][j] for j in selected), sum(graph[i]))
        state = 0 if share < Fraction(1, 3) else (1 if share < Fraction(2, 3) else 2)
        oracle[i][int(i in selected), state] += Fraction(1, len(allocations))
assert oracle == tables
print(f"PASS: {len(allocations)} complete allocations; {n * 6} exact arm probabilities matched.")

PASS: 15 complete allocations; 36 exact arm probabilities matched.


## 6. 边界与验收

补查 m=0、m=N、恰好 1/3 与 2/3、非法配额。完整测试文件还覆盖所有 N<=4 的图和 N=5..8 的多种图，
以及假设状态、重标号和 300 节点输入。

In [8]:
assert [exposure_state(k, 6) for k in (1, 2, 3, 4)] == [0, 1, 1, 2]
assert all(row[0, 0] == 1 for row in compute_support(graph, 0).values())
assert all(row[1, 2] == 1 for row in compute_support(graph, n).values())
try:
    compute_support(graph, n + 1)
except ValueError as error:
    print("Expected invalid-input rejection:", error)
else:
    raise AssertionError("Invalid quota was not rejected")
print("All demonstration assertions passed.")

Expected invalid-input rejection: m must be between 0 and 6
All demonstration assertions passed.


## 运行与适用范围

本目录包含设定说明、支持概率计算、识别规则、运行示例和测试代码。
从仓库根目录运行：

```bash
python -B -m unittest discover -s student_work/task05 -p "test_*.py" -v
python -B tools/run_student_tests.py
python -B -m unittest discover -s tests -p "test_identification_integration.py" -v
```

本实现仅处理均匀固定总配额 D2 与三档暴露 E2；需要简单无向、处理前固定的网络。
孤立节点报告不适用。结果用于数学与代码 review，未扩展共享四臂 backend 或训练 Estimation/Policy。
测试通过验证程序行为，不替代实验假设的领域审核。